## CS7DS2 - Optimisation Algorithms for Data Analysis - Final Assignment

Name: Swetha Sekar

Student ID: 25336453

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams.update({'font.size': 11})

## Benchmark Definitions

In [ ]:
# Benchmark A: Linear Regression Quadratic Loss
m = 1000
X_data = np.random.randn(m, 2)
theta_star = np.array([3.0, 4.0])
eps_noise = np.random.randn(m)
y_data = X_data @ theta_star + eps_noise

def loss_A(theta):
    r = X_data @ theta - y_data
    return 0.5 * np.mean(r**2)

def grad_A(theta):
    r = X_data @ theta - y_data
    return X_data.T @ r / m

def hessian_A_matrix():
    return X_data.T @ X_data / m

# Benchmark B: Toy Neural Network Quadratic Loss
def loss_B(x):
    return (x[0] - 1)**2 + 5*(x[1] - 2)**2 + np.sin(x[0])

def grad_B(x):
    return np.array([2*(x[0] - 1) + np.cos(x[0]), 10*(x[1] - 2)])

def hessian_B(x):
    return np.array([[2 - np.sin(x[0]), 0], [0, 10]])

# Benchmark C: Rosenbrock Function
def loss_C(x):
    return (1 - x[0])**2 + 100*(x[1] - x[0]**2)**2

def grad_C(x):
    g1 = -2*(1 - x[0]) - 400*x[0]*(x[1] - x[0]**2)
    g2 = 200*(x[1] - x[0]**2)
    return np.array([g1, g2])

def hessian_C(x):
    h11 = 2 + 1200*x[0]**2 - 400*x[1]
    h12 = -400*x[0]
    return np.array([[h11, h12], [h12, 200]])

# Starting points
theta0_A = np.array([0.0, 0.0])
x0_B = np.array([-1.0, 4.0])
x0_C = np.array([-1.0, 1.0])

## Optimiser Implementations

In [ ]:
def gradient_descent(grad_fn, loss_fn, x0, alpha, n_iters):
    x = x0.copy().astype(float)
    x_hist, f_hist = [x.copy()], [loss_fn(x)]
    for _ in range(n_iters):
        x = x - alpha * grad_fn(x)
        x_hist.append(x.copy())
        f_hist.append(loss_fn(x))
    return np.array(x_hist), np.array(f_hist)

def polyak_step(grad_fn, loss_fn, x0, f_star, eps, n_iters):
    x = x0.copy().astype(float)
    x_hist, f_hist, alpha_hist = [x.copy()], [loss_fn(x)], []
    for _ in range(n_iters):
        g = grad_fn(x)
        alpha_k = (loss_fn(x) - f_star) / (np.dot(g, g) + eps)
        alpha_hist.append(alpha_k)
        x = x - alpha_k * g
        x_hist.append(x.copy())
        f_hist.append(loss_fn(x))
    return np.array(x_hist), np.array(f_hist), np.array(alpha_hist)

def adagrad(grad_fn, loss_fn, x0, alpha0, eps, n_iters):
    x = x0.copy().astype(float)
    G = np.zeros_like(x)
    x_hist, f_hist, alpha_hist = [x.copy()], [loss_fn(x)], []
    for _ in range(n_iters):
        g = grad_fn(x)
        G += g**2
        eff_alpha = alpha0 / (np.sqrt(G) + eps)
        alpha_hist.append(np.mean(eff_alpha))
        x = x - eff_alpha * g
        x_hist.append(x.copy())
        f_hist.append(loss_fn(x))
    return np.array(x_hist), np.array(f_hist), np.array(alpha_hist)

def rmsprop(grad_fn, loss_fn, x0, alpha0, beta, eps, n_iters):
    x = x0.copy().astype(float)
    v = np.zeros_like(x)
    x_hist, f_hist, alpha_hist = [x.copy()], [loss_fn(x)], []
    for _ in range(n_iters):
        g = grad_fn(x)
        v = beta * v + (1 - beta) * g**2
        eff_alpha = alpha0 / (np.sqrt(v) + eps)
        alpha_hist.append(np.mean(eff_alpha))
        x = x - eff_alpha * g
        x_hist.append(x.copy())
        f_hist.append(loss_fn(x))
    return np.array(x_hist), np.array(f_hist), np.array(alpha_hist)

def heavy_ball(grad_fn, loss_fn, x0, alpha, beta, n_iters):
    x = x0.copy().astype(float)
    z = np.zeros_like(x)
    x_hist, f_hist = [x.copy()], [loss_fn(x)]
    for _ in range(n_iters):
        g = grad_fn(x)
        z = beta * z + alpha * g
        x = x - z
        x_hist.append(x.copy())
        f_hist.append(loss_fn(x))
    return np.array(x_hist), np.array(f_hist)

def nesterov_momentum(grad_fn, loss_fn, x0, alpha, beta_max, n_iters):
    x = x0.copy().astype(float)
    z = np.zeros_like(x)
    x_hist, f_hist = [x.copy()], [loss_fn(x)]
    for k in range(1, n_iters + 1):
        beta_k = min((k - 1) / (k + 2), beta_max)
        lookahead = x + beta_k * z
        g = grad_fn(lookahead)
        z = beta_k * z - alpha * g
        x = x + z
        x_hist.append(x.copy())
        f_hist.append(loss_fn(x))
    return np.array(x_hist), np.array(f_hist)

def adam_optimiser(grad_fn, loss_fn, x0, alpha, beta1, beta2, eps, n_iters):
    x = x0.copy().astype(float)
    m_vec, v_vec = np.zeros_like(x), np.zeros_like(x)
    x_hist, f_hist = [x.copy()], [loss_fn(x)]
    for t in range(1, n_iters + 1):
        g = grad_fn(x)
        m_vec = beta1 * m_vec + (1 - beta1) * g
        v_vec = beta2 * v_vec + (1 - beta2) * g**2
        m_hat = m_vec / (1 - beta1**t)
        v_hat = v_vec / (1 - beta2**t)
        x = x - alpha * m_hat / (np.sqrt(v_hat) + eps)
        x_hist.append(x.copy())
        f_hist.append(loss_fn(x))
    return np.array(x_hist), np.array(f_hist)

def newtons_method(grad_fn, hess_fn, loss_fn, x0, alpha, n_iters, damping=1e-8):
    x = x0.copy().astype(float)
    x_hist, f_hist, update_norms = [x.copy()], [loss_fn(x)], []
    for _ in range(n_iters):
        g = grad_fn(x)
        H = hess_fn(x) if callable(hess_fn) else hess_fn
        H_reg = H + damping * np.eye(len(x))
        try:
            p = np.linalg.solve(H_reg, g)
        except np.linalg.LinAlgError:
            p = g
        update_norms.append(np.linalg.norm(alpha * p))
        x = x - alpha * p
        x_hist.append(x.copy())
        f_hist.append(loss_fn(x))
    return np.array(x_hist), np.array(f_hist), np.array(update_norms)

## Contour Grid Setup

In [ ]:
x1_grid_B = np.linspace(-2, 3, 400)
x2_grid_B = np.linspace(-0.5, 5.5, 400)
X1B, X2B = np.meshgrid(x1_grid_B, x2_grid_B)
ZB = np.vectorize(lambda a, b: loss_B(np.array([a, b])))(X1B, X2B)

x1_grid_C = np.linspace(-1.5, 1.5, 400)
x2_grid_C = np.linspace(-0.5, 2.0, 400)
X1C, X2C = np.meshgrid(x1_grid_C, x2_grid_C)
ZC = np.vectorize(lambda a, b: loss_C(np.array([a, b])))(X1C, X2C)

def contour_B(ax):
    ax.contour(X1B, X2B, ZB, levels=30, cmap='viridis', alpha=0.7)

def contour_C(ax):
    ax.contour(X1C, X2C, ZC, levels=np.logspace(-1, 3.5, 30), cmap='viridis', alpha=0.7)

## Question 1: Adaptive Step-Size Methods [30 marks]

Comparing Polyak Step Size, Adagrad, RMSprop, and Heavy Ball (Polyak Momentum) against a constant step-size GD baseline across all three benchmarks.

In [ ]:
q1_results = {}
for bname, grad_fn, loss_fn, x0, params in [
    ('A', grad_A, loss_A, theta0_A, {
        'polyak': {'f_star': 0, 'eps': 1e-4},
        'adagrad': {'alpha0': 1.8, 'eps': 1e-5},
        'rmsprop': {'alpha0': 0.22, 'beta': 0.9, 'eps': 1e-5},
        'hb': {'alpha': 0.045, 'beta': 0.88},
        'gd': {'alpha': 0.08}
    }),
    ('B', grad_B, loss_B, x0_B, {
        'polyak': {'f_star': 0, 'eps': 1e-4},
        'adagrad': {'alpha0': 1.2, 'eps': 1e-5},
        'rmsprop': {'alpha0': 0.14, 'beta': 0.9, 'eps': 1e-5},
        'hb': {'alpha': 0.035, 'beta': 0.90},
        'gd': {'alpha': 0.06}
    }),
    ('C', grad_C, loss_C, x0_C, {
        'polyak': {'f_star': 0, 'eps': 1e-3},
        'adagrad': {'alpha0': 0.45, 'eps': 1e-5},
        'rmsprop': {'alpha0': 0.0035, 'beta': 0.9, 'eps': 1e-5},
        'hb': {'alpha': 0.0008, 'beta': 0.86},
        'gd': {'alpha': 0.0012}
    })
]:
    n = 120
    res = {}
    res['gd'] = gradient_descent(grad_fn, loss_fn, x0, params['gd']['alpha'], n)
    p = params['polyak']
    res['polyak'] = polyak_step(grad_fn, loss_fn, x0, p['f_star'], p['eps'], n)
    p = params['adagrad']
    res['adagrad'] = adagrad(grad_fn, loss_fn, x0, p['alpha0'], p['eps'], n)
    p = params['rmsprop']
    res['rmsprop'] = rmsprop(grad_fn, loss_fn, x0, p['alpha0'], p['beta'], p['eps'], n)
    p = params['hb']
    res['hb'] = heavy_ball(grad_fn, loss_fn, x0, p['alpha'], p['beta'], n)
    q1_results[bname] = res

### Q1(b): Objective Value vs Iteration

In [ ]:
title_map = {'A': 'Linear Regression', 'B': 'Toy Neural Network', 'C': 'Rosenbrock'}
for bname in ['A', 'B', 'C']:
    res = q1_results[bname]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.semilogy(res['gd'][1], 'k--', lw=1.5, label='GD (baseline)')
    ax.semilogy(res['polyak'][1], lw=2, label='Polyak Step')
    ax.semilogy(res['adagrad'][1], lw=2, label='Adagrad')
    ax.semilogy(res['rmsprop'][1], lw=2, label='RMSprop')
    ax.semilogy(res['hb'][1], lw=2, label='Heavy Ball')
    ax.set_xlabel('Iteration'); ax.set_ylabel('Objective Value (log scale)')
    ax.set_title(f'Q1: Convergence — Benchmark {bname} ({title_map[bname]})')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

**Benchmark A:** All methods converge on the well-conditioned quadratic surface. Polyak converges fastest by scaling the step with the optimality gap. Adagrad and RMSprop adapt per-coordinate, handling the moderate curvature well. Heavy Ball benefits from consistent gradient direction.

**Benchmark B:** The $\sin(x_1)$ introduces mild non-convexity. RMSprop performs well due to per-coordinate adaptation to the anisotropic curvature (2:10 eigenvalue ratio).

**Benchmark C:** Extreme ill-conditioning slows all methods. The narrow curved valley demands small steps. RMSprop and Adagrad partially compensate through per-coordinate adaptation.

### Q1(c): Contour Plots with Trajectories

In [ ]:
for bname, contour_fn in [('B', contour_B), ('C', contour_C)]:
    res = q1_results[bname]
    fig, ax = plt.subplots(figsize=(8, 7))
    contour_fn(ax)
    labels = {'gd': 'GD', 'polyak': 'Polyak', 'adagrad': 'Adagrad', 'rmsprop': 'RMSprop', 'hb': 'Heavy Ball'}
    styles = {'gd': ('k', '--'), 'polyak': ('tab:blue', '-'), 'adagrad': ('tab:orange', '-'),
              'rmsprop': ('tab:green', '-'), 'hb': ('tab:red', '-')}
    for method in ['gd', 'polyak', 'adagrad', 'rmsprop', 'hb']:
        hist = res[method][0]
        c, ls = styles[method]
        ax.plot(hist[:, 0], hist[:, 1], color=c, linestyle=ls, lw=1.5, alpha=0.8, label=labels[method])
    x0_used = x0_B if bname == 'B' else x0_C
    ax.plot(*x0_used, 'k*', ms=14, label='Start')
    opt = [0.61, 2.0] if bname == 'B' else [1.0, 1.0]
    ax.plot(*opt, 'r*', ms=14, label='Optimum')
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.set_title(f'Q1: Trajectories — Benchmark {bname} ({title_map[bname]})')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.2)
    plt.tight_layout(); plt.show()

**Benchmark B:** Elliptical contours reflect the 5:1 curvature ratio. Adaptive methods navigate more efficiently. Heavy Ball's momentum causes slight overshoot.

**Benchmark C:** The banana-shaped valley is visible. Methods first descend quickly towards the valley then slowly follow the floor towards (1,1).

### Q1(d): Adaptive Step-Size Evolution

In [ ]:
for bname in ['A', 'B', 'C']:
    res = q1_results[bname]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(res['polyak'][2], lw=2, label='Polyak')
    ax.plot(res['adagrad'][2], lw=2, label='Adagrad (mean eff.)')
    ax.plot(res['rmsprop'][2], lw=2, label='RMSprop (mean eff.)')
    ax.set_xlabel('Iteration'); ax.set_ylabel('Effective Step Size')
    ax.set_title(f'Q1: Step-Size Evolution — Benchmark {bname} ({title_map[bname]})')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

* Polyak step decreases as the iterate approaches the minimum (numerator $f(x_k) - f^*$ shrinks).
* Adagrad's effective step decays monotonically as accumulated gradient norm grows.
* RMSprop maintains a more stable effective step due to exponential forgetting ($\beta = 0.9$).

## Question 2: Momentum and Stochastic Methods [30 marks]

Comparing Nesterov Momentum, Adam, Mini-Batch SGD, and SGD with Noise.

In [ ]:
q2_results = {}
for bname, grad_fn, loss_fn, x0, params in [
    ('A', grad_A, loss_A, theta0_A, {
        'nesterov': {'alpha': 0.06, 'beta_max': 0.90},
        'adam': {'alpha': 0.12, 'beta1': 0.82, 'beta2': 0.999, 'eps': 1e-8},
        'gd': {'alpha': 0.08}
    }),
    ('B', grad_B, loss_B, x0_B, {
        'nesterov': {'alpha': 0.035, 'beta_max': 0.92},
        'adam': {'alpha': 0.08, 'beta1': 0.80, 'beta2': 0.999, 'eps': 1e-8},
        'gd': {'alpha': 0.06}
    }),
    ('C', grad_C, loss_C, x0_C, {
        'nesterov': {'alpha': 0.0007, 'beta_max': 0.90},
        'adam': {'alpha': 0.006, 'beta1': 0.80, 'beta2': 0.999, 'eps': 1e-8},
        'gd': {'alpha': 0.0012}
    })
]:
    n = 150
    res = {}
    res['gd'] = gradient_descent(grad_fn, loss_fn, x0, params['gd']['alpha'], n)
    p = params['nesterov']
    res['nesterov'] = nesterov_momentum(grad_fn, loss_fn, x0, p['alpha'], p['beta_max'], n)
    p = params['adam']
    res['adam'] = adam_optimiser(grad_fn, loss_fn, x0, p['alpha'], p['beta1'], p['beta2'], p['eps'], n)
    q2_results[bname] = res

### Q2(a,b): Convergence Comparison

In [ ]:
for bname in ['A', 'B', 'C']:
    res = q2_results[bname]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.semilogy(res['gd'][1], 'k--', lw=1.5, label='GD (baseline)')
    ax.semilogy(res['nesterov'][1], lw=2, label='Nesterov Momentum')
    ax.semilogy(res['adam'][1], lw=2, label='Adam')
    ax.set_xlabel('Iteration'); ax.set_ylabel('Objective Value (log scale)')
    ax.set_title(f'Q2: Convergence — Benchmark {bname} ({title_map[bname]})')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

Both Nesterov and Adam outperform the GD baseline across all benchmarks. Adam's per-coordinate adaptive scaling handles the anisotropic curvature of Benchmarks B and C particularly well. Nesterov's lookahead correction prevents overshooting.

### Q2(c): Contour Trajectories

In [ ]:
for bname, contour_fn in [('B', contour_B), ('C', contour_C)]:
    res = q2_results[bname]
    fig, ax = plt.subplots(figsize=(8, 7))
    contour_fn(ax)
    for method, lab, c in [('gd', 'GD', 'k'), ('nesterov', 'Nesterov', 'tab:blue'), ('adam', 'Adam', 'tab:red')]:
        hist = res[method][0]
        ls = '--' if method == 'gd' else '-'
        ax.plot(hist[:, 0], hist[:, 1], color=c, linestyle=ls, lw=1.5, alpha=0.8, label=lab)
    x0_used = x0_B if bname == 'B' else x0_C
    ax.plot(*x0_used, 'k*', ms=14, label='Start')
    opt = [0.61, 2.0] if bname == 'B' else [1.0, 1.0]
    ax.plot(*opt, 'r*', ms=14, label='Optimum')
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.set_title(f'Q2: Trajectories — Benchmark {bname} ({title_map[bname]})')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.2)
    plt.tight_layout(); plt.show()

### Q2(d): Mini-Batch SGD — Batch Size Effect

In [ ]:
def mini_batch_sgd(X, y, theta0, alpha, batch_size, n_epochs, seed=42):
    rng = np.random.RandomState(seed)
    theta = theta0.copy().astype(float)
    n = len(y)
    loss_fn_local = lambda th: 0.5 * np.mean((X @ th - y)**2)
    epoch_losses = [loss_fn_local(theta)]
    for _ in range(n_epochs):
        idx = rng.permutation(n)
        for i in range(0, n, batch_size):
            batch_idx = idx[i:i+batch_size]
            Xb, yb = X[batch_idx], y[batch_idx]
            r = Xb @ theta - yb
            g = Xb.T @ r / len(yb)
            theta = theta - alpha * g
        epoch_losses.append(loss_fn_local(theta))
    return np.array(epoch_losses)

sgd_b5 = mini_batch_sgd(X_data, y_data, theta0_A, 0.06, 5, 50)
sgd_b40 = mini_batch_sgd(X_data, y_data, theta0_A, 0.06, 40, 50)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(sgd_b5, lw=2, label='SGD (b=5)')
ax.semilogy(sgd_b40, lw=2, label='SGD (b=40)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss (log scale)')
ax.set_title('Q2: Mini-Batch SGD — Batch Size Effect (Benchmark A)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

With b=5, each epoch does 200 updates (higher gradient noise but more steps). With b=40, only 25 updates per epoch (lower noise, fewer steps). Smaller batch converges faster per epoch but oscillates more around the minimum.

### Q2(e): SGD with Increased Noise

In [ ]:
y_noisy = X_data @ theta_star + 6.0 * eps_noise

sgd_noisy_b5 = mini_batch_sgd(X_data, y_noisy, theta0_A, 0.06, 5, 50)
sgd_noisy_b40 = mini_batch_sgd(X_data, y_noisy, theta0_A, 0.06, 40, 50)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].semilogy(sgd_b5, lw=2, label='b=5'); axes[0].semilogy(sgd_b40, lw=2, label='b=40')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss (log scale)')
axes[0].set_title('Original Noise ($\\sigma=1$)'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].semilogy(sgd_noisy_b5, lw=2, label='b=5'); axes[1].semilogy(sgd_noisy_b40, lw=2, label='b=40')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss (log scale)')
axes[1].set_title('High Noise ($\\sigma=6$)'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

Increasing noise by factor 6 raises the irreducible loss floor by factor ~36. Both batch sizes converge to a higher minimum loss. The relative advantage of smaller batch sizes diminishes under higher noise, as gradient noise already dominates regardless of batch size.

## Question 3: Newton's Method and Local Approximation [30 marks]

### Q3(a): Local Approximations of $g(x) = x^4$

In [ ]:
x_plot = np.linspace(-0.5, 1.0, 400)
x0_approx = 0.25
g0 = x0_approx**4
gp0 = 4*x0_approx**3
gpp0 = 12*x0_approx**2

first_order = g0 + gp0 * (x_plot - x0_approx)
second_order = g0 + gp0 * (x_plot - x0_approx) + 0.5 * gpp0 * (x_plot - x0_approx)**2

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(x_plot, x_plot**4, 'k-', lw=2.5, label='$g(x) = x^4$')
ax.plot(x_plot, first_order, 'b--', lw=2, label='First-order approximation')
ax.plot(x_plot, second_order, 'r-.', lw=2, label='Second-order approximation')
ax.plot(x0_approx, g0, 'ko', ms=10, zorder=5, label=f'$x_0 = {x0_approx}$')
ax.set_xlabel('$x$'); ax.set_ylabel('$g(x)$')
ax.set_title('Q3: Local Approximations of $g(x) = x^4$ at $x_0 = 0.25$')
ax.set_ylim(-0.1, 0.6); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

The first-order approximation (tangent line) captures only the local slope. The second-order approximation (parabola) captures curvature and provides a better local fit. The Newton update moves to the minimum of this parabola: $x_{k+1} = x_k - g'(x_k)/g''(x_k)$.

### Q3(b,c): Newton vs GD on All Benchmarks

In [ ]:
H_A = hessian_A_matrix()
hess_A_fn = lambda x: H_A

q3_results = {}
for bname, grad_fn, hess_fn, loss_fn, x0, params in [
    ('A', grad_A, hess_A_fn, loss_A, theta0_A, {'gd_alpha': 0.08, 'newton_alpha': 1.0}),
    ('B', grad_B, hessian_B, loss_B, x0_B, {'gd_alpha': 0.06, 'newton_alpha': 0.85}),
    ('C', grad_C, hessian_C, loss_C, x0_C, {'gd_alpha': 0.001, 'newton_alpha': 0.22})
]:
    res = {}
    res['gd'] = gradient_descent(grad_fn, loss_fn, x0, params['gd_alpha'], 80)
    gd_norms = [np.linalg.norm(res['gd'][0][i+1] - res['gd'][0][i]) for i in range(len(res['gd'][0])-1)]
    res['gd_norms'] = np.array(gd_norms)
    res['newton'] = newtons_method(grad_fn, hess_fn, loss_fn, x0, params['newton_alpha'], 20, damping=1e-8)
    q3_results[bname] = res

for bname in ['A', 'B', 'C']:
    res = q3_results[bname]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.semilogy(res['gd'][1], 'b-', lw=2, label='Gradient Descent (80 iters)')
    ax.semilogy(res['newton'][1], 'r-', lw=2, label="Newton's Method (20 iters)")
    ax.set_xlabel('Iteration'); ax.set_ylabel('Objective Value (log scale)')
    ax.set_title(f'Q3: Newton vs GD — Benchmark {bname} ({title_map[bname]})')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

**Benchmark A:** Newton converges in one step (exact for quadratic loss). **Benchmark B:** Newton converges in few iterations due to near-quadratic structure. **Benchmark C:** Even on ill-conditioned Rosenbrock, Newton dramatically outperforms GD by using curvature information.

### Q3(d): Contour Trajectories

In [ ]:
for bname, contour_fn in [('B', contour_B), ('C', contour_C)]:
    res = q3_results[bname]
    fig, ax = plt.subplots(figsize=(8, 7))
    contour_fn(ax)
    ax.plot(res['gd'][0][:, 0], res['gd'][0][:, 1], 'b-o', ms=3, lw=1.5, alpha=0.8, label='GD')
    ax.plot(res['newton'][0][:, 0], res['newton'][0][:, 1], 'r-s', ms=5, lw=2, alpha=0.9, label='Newton')
    x0_used = x0_B if bname == 'B' else x0_C
    ax.plot(*x0_used, 'k*', ms=14, label='Start')
    opt = [0.61, 2.0] if bname == 'B' else [1.0, 1.0]
    ax.plot(*opt, 'g*', ms=14, label='Optimum')
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.set_title(f'Q3: Newton vs GD Trajectories — Benchmark {bname}')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.2)
    plt.tight_layout(); plt.show()

### Q3(e): Update Magnitude vs Iteration

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
short_names = {'A': 'Linear Reg.', 'B': 'Toy NN', 'C': 'Rosenbrock'}
for idx, bname in enumerate(['A', 'B', 'C']):
    res = q3_results[bname]
    ax = axes[idx]
    ax.semilogy(res['gd_norms'], 'b-', lw=2, label='GD')
    ax.semilogy(res['newton'][2], 'r-', lw=2, label='Newton')
    ax.set_xlabel('Iteration'); ax.set_ylabel('Update Magnitude (log scale)')
    ax.set_title(f'Benchmark {bname} ({short_names[bname]})')
    ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('Q3: Update Magnitude vs Iteration', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

Newton's update magnitude decreases rapidly (quadratic convergence), while GD's decreases linearly. For Benchmark A, Newton's update drops to near-zero after one step.

## Question 4: Derivative Approximation and Derivative-Free Optimisation [30 marks]

### Q4 Part A: Finite Difference and Nesterov Random Search (Benchmark B)

In [ ]:
def fd_gradient(loss_fn, x, delta):
    n = len(x)
    g = np.zeros(n)
    f0 = loss_fn(x)
    for i in range(n):
        ei = np.zeros(n); ei[i] = 1.0
        g[i] = (loss_fn(x + delta * ei) - f0) / delta
    return g

def fd_gradient_descent(loss_fn, x0, alpha, delta, n_iters):
    x = x0.copy().astype(float)
    x_hist, f_hist = [x.copy()], [loss_fn(x)]
    for _ in range(n_iters):
        g = fd_gradient(loss_fn, x, delta)
        x = x - alpha * g
        x_hist.append(x.copy())
        f_hist.append(loss_fn(x))
    return np.array(x_hist), np.array(f_hist)

def nesterov_random_search(loss_fn, x0, alpha, delta, n_iters, seed=42):
    rng = np.random.RandomState(seed)
    x = x0.copy().astype(float)
    x_hist, f_hist = [x.copy()], [loss_fn(x)]
    for _ in range(n_iters):
        u = rng.randn(len(x))
        u = u / np.linalg.norm(u)
        df = (loss_fn(x + delta * u) - loss_fn(x)) / delta
        x = x - alpha * df * u
        x_hist.append(x.copy())
        f_hist.append(loss_fn(x))
    return np.array(x_hist), np.array(f_hist)

gd_exact_B = gradient_descent(grad_B, loss_B, x0_B, 0.06, 220)
fd_good_B = fd_gradient_descent(loss_B, x0_B, 0.08, 0.05, 120)
fd_poor_B = fd_gradient_descent(loss_B, x0_B, 0.08, 0.8, 120)
nrs_B = nesterov_random_search(loss_B, x0_B, 0.025, 0.08, 220)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(gd_exact_B[1][:121], 'k-', lw=2, label='Exact GD ($\\alpha=0.06$)')
ax.semilogy(fd_good_B[1], 'b-', lw=2, label='FD GD ($\\delta=0.05$, good)')
ax.semilogy(fd_poor_B[1], 'r-', lw=2, label='FD GD ($\\delta=0.8$, poor)')
ax.semilogy(nrs_B[1], 'g-', lw=1.5, alpha=0.8, label='Nesterov Random Search')
ax.set_xlabel('Iteration'); ax.set_ylabel('Objective Value (log scale)')
ax.set_title('Q4A: Derivative Approximation Methods — Benchmark B')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
contour_B(ax)
ax.plot(gd_exact_B[0][:121, 0], gd_exact_B[0][:121, 1], 'k-', lw=1.5, label='Exact GD')
ax.plot(fd_good_B[0][:, 0], fd_good_B[0][:, 1], 'b-', lw=1.5, alpha=0.8, label='FD ($\\delta=0.05$)')
ax.plot(fd_poor_B[0][:, 0], fd_poor_B[0][:, 1], 'r-', lw=1.5, alpha=0.8, label='FD ($\\delta=0.8$)')
ax.plot(nrs_B[0][:, 0], nrs_B[0][:, 1], 'g-', lw=1, alpha=0.6, label='Nesterov Random')
ax.plot(*x0_B, 'k*', ms=14, label='Start')
ax.plot(0.61, 2.0, 'r*', ms=14, label='Optimum')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('Q4A: Trajectories on Benchmark B')
ax.legend(fontsize=9); ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.show()

Exact GD converges fastest. FD with good $\delta=0.05$ closely tracks exact GD, confirming the gradient approximation quality. Poor $\delta=0.8$ introduces large truncation errors. Nesterov random search converges slowest due to its stochastic nature.

### Q4 Part B: Nelder-Mead and Grid Search (Benchmark C)

In [ ]:
def nelder_mead(loss_fn, x0, step, n_iters, alpha_r=1.0, gamma=2.0, rho=0.5, sigma=0.5):
    n = len(x0)
    simplex = np.zeros((n + 1, n))
    simplex[0] = x0.copy()
    for i in range(n):
        simplex[i + 1] = x0.copy()
        simplex[i + 1][i] += step
    f_vals = np.array([loss_fn(v) for v in simplex])
    centroid_hist = [np.mean(simplex, axis=0).copy()]
    f_best_hist = [np.min(f_vals)]
    for _ in range(n_iters):
        order = np.argsort(f_vals)
        simplex = simplex[order]; f_vals = f_vals[order]
        centroid = np.mean(simplex[:-1], axis=0)
        x_r = centroid + alpha_r * (centroid - simplex[-1])
        f_r = loss_fn(x_r)
        if f_vals[0] <= f_r < f_vals[-2]:
            simplex[-1] = x_r; f_vals[-1] = f_r
        elif f_r < f_vals[0]:
            x_e = centroid + gamma * (centroid - simplex[-1])
            f_e = loss_fn(x_e)
            if f_e < f_r:
                simplex[-1] = x_e; f_vals[-1] = f_e
            else:
                simplex[-1] = x_r; f_vals[-1] = f_r
        else:
            if f_r < f_vals[-1]:
                x_c = centroid + rho * (x_r - centroid)
                f_c = loss_fn(x_c)
                if f_c <= f_r:
                    simplex[-1] = x_c; f_vals[-1] = f_c
                else:
                    for i in range(1, n+1):
                        simplex[i] = simplex[0] + sigma*(simplex[i]-simplex[0])
                        f_vals[i] = loss_fn(simplex[i])
            else:
                x_c = centroid + rho * (simplex[-1] - centroid)
                f_c = loss_fn(x_c)
                if f_c < f_vals[-1]:
                    simplex[-1] = x_c; f_vals[-1] = f_c
                else:
                    for i in range(1, n+1):
                        simplex[i] = simplex[0] + sigma*(simplex[i]-simplex[0])
                        f_vals[i] = loss_fn(simplex[i])
        centroid_hist.append(np.mean(simplex, axis=0).copy())
        f_best_hist.append(np.min(f_vals))
    return np.array(centroid_hist), np.array(f_best_hist), simplex

nm_hist, nm_fvals, nm_final = nelder_mead(loss_C, x0_C, 0.35, 160)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
contour_C(axes[0])
axes[0].plot(nm_hist[:, 0], nm_hist[:, 1], 'b-o', ms=2, lw=1.5, alpha=0.8, label='Nelder-Mead centroid')
axes[0].plot(*x0_C, 'k*', ms=14, label='Start')
axes[0].plot(1.0, 1.0, 'r*', ms=14, label='Optimum (1,1)')
axes[0].set_xlabel('$x_1$'); axes[0].set_ylabel('$x_2$')
axes[0].set_title('Q4B: Nelder-Mead Trajectory — Rosenbrock')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.2)

axes[1].semilogy(nm_fvals, 'b-', lw=2)
axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('Best Objective (log scale)')
axes[1].set_title('Q4B: Nelder-Mead Convergence')
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
x1_gs = np.linspace(-2, 2, 55)
x2_gs = np.linspace(-1, 3, 55)
X1_GS, X2_GS = np.meshgrid(x1_gs, x2_gs)
Z_GS = np.vectorize(lambda a, b: loss_C(np.array([a, b])))(X1_GS, X2_GS)

best_idx = np.unravel_index(Z_GS.argmin(), Z_GS.shape)
best_point = np.array([X1_GS[best_idx], X2_GS[best_idx]])
best_val = Z_GS[best_idx]
best_so_far = np.minimum.accumulate(Z_GS.flatten())

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
contour_C(axes[0])
axes[0].scatter(X1_GS.flatten(), X2_GS.flatten(), s=3, c='gray', alpha=0.4, label='Grid points')
axes[0].plot(*best_point, 'r*', ms=16, zorder=5, label=f'Best: ({best_point[0]:.2f}, {best_point[1]:.2f})')
axes[0].plot(1.0, 1.0, 'g*', ms=14, zorder=5, label='True optimum (1,1)')
axes[0].set_xlabel('$x_1$'); axes[0].set_ylabel('$x_2$')
axes[0].set_title('Q4B: Grid Search Sampling — Rosenbrock')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.2)

axes[1].semilogy(best_so_far, 'b-', lw=1.5)
axes[1].set_xlabel('Grid Point Index'); axes[1].set_ylabel('Best Value So Far (log scale)')
axes[1].set_title(f'Q4B: Grid Search Best-So-Far (best f = {best_val:.4f})')
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

Nelder-Mead navigates the Rosenbrock valley without gradient information, demonstrating its effectiveness in low-dimensional non-convex settings. Grid search guarantees coverage within the search region but is limited by grid resolution and scales poorly with dimension.

## Question 5: Constrained Optimisation [30 marks]

Benchmark B with constraint $x_1 \geq 0.5$, starting from infeasible point $x_0 = (0.2, 4.0)$.

In [ ]:
x0_q5 = np.array([0.2, 4.0])

def project_q5(x):
    x_p = x.copy()
    x_p[0] = max(0.5, x_p[0])
    return x_p

def penalty_loss_q5(x, lam):
    return loss_B(x) + lam * max(0, -x[0] + 0.5)

def penalty_grad_q5(x, lam):
    g = grad_B(x).copy()
    if x[0] < 0.5:
        g[0] -= lam
    return g

def projected_gd_q5(x0, alpha, n_iters):
    x = x0.copy().astype(float)
    x_hist, f_hist = [x.copy()], [loss_B(x)]
    for _ in range(n_iters):
        g = grad_B(x)
        x = project_q5(x - alpha * g)
        x_hist.append(x.copy())
        f_hist.append(loss_B(x))
    return np.array(x_hist), np.array(f_hist)

def penalty_gd_q5(x0, alpha, lam, n_iters):
    x = x0.copy().astype(float)
    x_hist, f_hist = [x.copy()], [penalty_loss_q5(x, lam)]
    for _ in range(n_iters):
        g = penalty_grad_q5(x, lam)
        x = x - alpha * g
        x_hist.append(x.copy())
        f_hist.append(penalty_loss_q5(x, lam))
    return np.array(x_hist), np.array(f_hist)

gd_uncons = gradient_descent(grad_B, loss_B, x0_q5, 0.07, 100)
pgd_q5 = projected_gd_q5(x0_q5, 0.08, 100)
pen_015 = penalty_gd_q5(x0_q5, 0.05, 0.15, 100)
pen_18 = penalty_gd_q5(x0_q5, 0.05, 1.8, 100)
pen_45 = penalty_gd_q5(x0_q5, 0.03, 4.5, 100)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy([loss_B(x) for x in gd_uncons[0]], 'k--', lw=1.5, label='Unconstrained GD')
ax.semilogy(pgd_q5[1], 'b-', lw=2, label='Projected GD')
ax.semilogy([loss_B(x) for x in pen_015[0]], 'g-', lw=2, label='Penalty ($\\lambda=0.15$)')
ax.semilogy([loss_B(x) for x in pen_18[0]], color='orange', lw=2, label='Penalty ($\\lambda=1.8$)')
ax.semilogy([loss_B(x) for x in pen_45[0]], 'r-', lw=2, label='Penalty ($\\lambda=4.5$)')
ax.set_xlabel('Iteration'); ax.set_ylabel('$f(x)$ (log scale)')
ax.set_title('Q5: Objective Value vs Iteration')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
x1_q5 = np.linspace(-0.5, 3.0, 400)
x2_q5 = np.linspace(0.0, 5.5, 400)
X1Q5, X2Q5 = np.meshgrid(x1_q5, x2_q5)
ZQ5 = np.vectorize(lambda a, b: loss_B(np.array([a, b])))(X1Q5, X2Q5)

fig, ax = plt.subplots(figsize=(8, 7))
ax.contour(X1Q5, X2Q5, ZQ5, levels=30, cmap='viridis', alpha=0.7)
ax.axvline(x=0.5, color='red', lw=2, linestyle='--', label='$x_1 = 0.5$')
ax.fill_betweenx([0, 5.5], -0.5, 0.5, alpha=0.15, color='red', label='Infeasible')
ax.plot(gd_uncons[0][:, 0], gd_uncons[0][:, 1], 'k--', lw=1.5, alpha=0.7, label='Unconstrained')
ax.plot(pgd_q5[0][:, 0], pgd_q5[0][:, 1], 'b-o', ms=2, lw=1.5, label='Projected GD')
ax.plot(pen_015[0][:, 0], pen_015[0][:, 1], 'g-', lw=1.5, alpha=0.8, label='Penalty $\\lambda=0.15$')
ax.plot(pen_18[0][:, 0], pen_18[0][:, 1], color='orange', lw=1.5, alpha=0.8, label='Penalty $\\lambda=1.8$')
ax.plot(pen_45[0][:, 0], pen_45[0][:, 1], 'r-', lw=1.5, alpha=0.8, label='Penalty $\\lambda=4.5$')
ax.plot(*x0_q5, 'k*', ms=14, label='Start (0.2, 4.0)')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('Q5: Contour with Feasible Boundary and Trajectories')
ax.legend(fontsize=8, loc='upper right'); ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.show()

### Q5: Constraint Violation Analysis

In [ ]:
def violation(x_hist):
    return np.array([max(0, 0.5 - x[0]) for x in x_hist])

v_uncons = violation(gd_uncons[0])
v_pgd = violation(pgd_q5[0])
v_pen015 = violation(pen_015[0])
v_pen18 = violation(pen_18[0])
v_pen45 = violation(pen_45[0])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax_idx, (ax, lim) in enumerate(zip(axes, [101, 31])):
    for v, lab, c, ls in [
        (v_uncons[:lim], 'Unconstrained', 'k', '--'),
        (v_pgd[:lim], 'Projected GD', 'b', '-'),
        (v_pen015[:lim], 'Penalty $\\lambda=0.15$', 'g', '-'),
        (v_pen18[:lim], 'Penalty $\\lambda=1.8$', 'orange', '-'),
        (v_pen45[:lim], 'Penalty $\\lambda=4.5$', 'r', '-')
    ]:
        mask = v > 0
        if np.any(mask):
            ax.semilogy(np.where(mask)[0], v[mask], color=c, linestyle=ls, lw=2, label=lab)
    ax.set_xlabel('Iteration'); ax.set_ylabel('Constraint Violation (log scale)')
    title = 'Full View' if ax_idx == 0 else 'First 30 Iterations (Zoomed)'
    ax.set_title(f'Q5: Constraint Violation — {title}')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

Projected GD achieves feasibility in one step (projection directly enforces $x_1 \geq 0.5$). Penalty methods enforce feasibility gradually: $\lambda=4.5$ is fastest, $\lambda=0.15$ is slowest. The trade-off: projection gives hard feasibility guarantees but requires a projection oracle; penalty methods only need gradient modifications.

## Question 6: Linear Programmes and Frank-Wolfe [30 marks]

In [ ]:
X_bounds = [(0.5, 5.0), (-5.0, 10.0)]

def fw_lp_box(grad, bounds):
    z = np.zeros(len(grad))
    for i in range(len(grad)):
        z[i] = bounds[i][0] if grad[i] > 0 else bounds[i][1]
    return z

def frank_wolfe(grad_fn, loss_fn, x0, bounds, beta, n_iters):
    x = x0.copy().astype(float)
    x_hist, f_hist, z_hist = [x.copy()], [loss_fn(x)], []
    for _ in range(n_iters):
        g = grad_fn(x)
        z = fw_lp_box(g, bounds)
        z_hist.append(z.copy())
        x = beta * x + (1 - beta) * z
        x_hist.append(x.copy())
        f_hist.append(loss_fn(x))
    return np.array(x_hist), np.array(f_hist), np.array(z_hist)

### Q6(I): Linear Programme

In [ ]:
x1_q6 = np.linspace(0, 5.5, 300)
x2_q6 = np.linspace(-6, 11, 300)
X1Q6, X2Q6 = np.meshgrid(x1_q6, x2_q6)

fig, ax = plt.subplots(figsize=(8, 7))
cs = ax.contour(X1Q6, X2Q6, X1Q6 + 2*X2Q6, levels=20, cmap='coolwarm', alpha=0.7)
ax.clabel(cs, inline=True, fontsize=8)
rect = plt.Rectangle((0.5, -5), 4.5, 15, fill=True, facecolor='lightblue', edgecolor='black',
                      alpha=0.3, lw=2, label='Feasible region')
ax.add_patch(rect)
ax.plot(0.5, -5, 'r*', ms=16, zorder=5, label='LP optimum (0.5, -5)')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('Q6-I: LP — $f(x) = x_1 + 2x_2$, $f^* = -9.5$')
ax.legend(fontsize=9); ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.show()

print(f'LP solution: x* = (0.5, -5), f* = {0.5 + 2*(-5)}')

Since both coefficients in $a = [1, 2]^T$ are positive, the LP minimum over the box constraint is at the lower bounds: $x^* = (0.5, -5)$ with $f^* = -9.5$. This illustrates that LP optima on polytopes occur at vertices.

### Q6(II): Frank-Wolfe — Interior Optimum

In [ ]:
def loss_q6_int(x): return (x[0]-1)**2 + (x[1]-5)**2
def grad_q6_int(x): return np.array([2*(x[0]-1), 2*(x[1]-5)])

fw_int_090 = frank_wolfe(grad_q6_int, loss_q6_int, np.array([1.0, 1.0]), X_bounds, 0.90, 180)
fw_int_0985 = frank_wolfe(grad_q6_int, loss_q6_int, np.array([1.0, 1.0]), X_bounds, 0.985, 180)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(fw_int_090[1], 'b-', lw=2, label='FW $\\beta=0.90$')
ax.semilogy(fw_int_0985[1], 'r-', lw=2, label='FW $\\beta=0.985$')
ax.set_xlabel('Iteration'); ax.set_ylabel('Objective (log scale)')
ax.set_title('Q6-II: Frank-Wolfe Convergence — Interior Optimum')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

### Q6(III): Frank-Wolfe — Boundary Optimum

In [ ]:
def loss_q6_bnd(x): return x[0]**2 + x[1]**2
def grad_q6_bnd(x): return np.array([2*x[0], 2*x[1]])

fw_bnd = frank_wolfe(grad_q6_bnd, loss_q6_bnd, np.array([3.0, 3.0]), X_bounds, 0.93, 140)

ZQ6_INT = np.vectorize(lambda a,b: loss_q6_int(np.array([a,b])))(X1Q6, X2Q6)
ZQ6_BND = np.vectorize(lambda a,b: loss_q6_bnd(np.array([a,b])))(X1Q6, X2Q6)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

ax = axes[0]
ax.contour(X1Q6, X2Q6, ZQ6_INT, levels=20, cmap='viridis', alpha=0.7)
rect1 = plt.Rectangle((0.5,-5),4.5,15,fill=True,facecolor='lightblue',edgecolor='black',alpha=0.2,lw=2)
ax.add_patch(rect1)
ax.plot(fw_int_090[0][:,0], fw_int_090[0][:,1], 'b-o', ms=2, lw=1.5, label='FW $\\beta=0.90$')
ax.plot(fw_int_0985[0][:,0], fw_int_0985[0][:,1], 'r-o', ms=2, lw=1.5, label='FW $\\beta=0.985$')
ax.plot(1.0, 1.0, 'k*', ms=14, label='Start'); ax.plot(1.0, 5.0, 'g*', ms=14, label='Optimum (1,5)')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('Q6-II: Interior Optimum'); ax.legend(fontsize=8); ax.grid(True, alpha=0.2)

ax = axes[1]
ax.contour(X1Q6, X2Q6, ZQ6_BND, levels=20, cmap='viridis', alpha=0.7)
rect2 = plt.Rectangle((0.5,-5),4.5,15,fill=True,facecolor='lightblue',edgecolor='black',alpha=0.2,lw=2)
ax.add_patch(rect2)
ax.plot(fw_bnd[0][:,0], fw_bnd[0][:,1], 'b-o', ms=2, lw=1.5, label='FW $\\beta=0.93$')
ax.plot(3.0, 3.0, 'k*', ms=14, label='Start'); ax.plot(0.5, 0.0, 'g*', ms=14, label='Opt (0.5,0)')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('Q6-III: Boundary Optimum'); ax.legend(fontsize=8); ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.show()

### Q6: Evolution of $x_k$ and $z_k$

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax_idx, (data, title) in enumerate([
    (fw_int_090, 'Interior ($\\beta=0.90$)'),
    (fw_int_0985, 'Interior ($\\beta=0.985$)'),
    (fw_bnd, 'Boundary ($\\beta=0.93$)')
]):
    ax = axes[ax_idx // 2, ax_idx % 2]
    ax.plot(data[0][:,0], 'b-', lw=1.5, label='$x_1^{(k)}$')
    ax.plot(data[0][:,1], 'b--', lw=1.5, label='$x_2^{(k)}$')
    ax.plot(range(1,len(data[2])+1), data[2][:,0], 'r:', lw=1.5, label='$z_1^{(k)}$')
    ax.plot(range(1,len(data[2])+1), data[2][:,1], 'r-.', lw=1.5, label='$z_2^{(k)}$')
    ax.set_xlabel('Iteration'); ax.set_ylabel('Value')
    ax.set_title(title); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.semilogy(fw_int_090[1], 'b-', lw=2, label='Interior $\\beta=0.90$')
ax.semilogy(fw_int_0985[1], 'r-', lw=2, label='Interior $\\beta=0.985$')
ax.semilogy(fw_bnd[1], 'g-', lw=2, label='Boundary $\\beta=0.93$')
ax.set_xlabel('Iteration'); ax.set_ylabel('Objective (log)')
ax.set_title('Convergence Comparison'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('Q6: Evolution of $x_k$ and $z_k$', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

**Interior optimum:** $z_k$ oscillates between vertices of $\mathcal{X}$, while $x_k$ converges smoothly via convex combinations. Smaller $\beta$ (0.90) converges faster but with more oscillation in $z_k$.

**Boundary optimum:** $x_k$ converges to $(0.5, 0)$ on the boundary. Frank-Wolfe naturally handles boundary optima since the LP subproblem favours extreme points, and the iterate settles at the boundary without explicit projection.